## Exploration de l'API SNCF Open Data : Dataset Transilien

## Contexte du dataset

Le dataset **Comptage des voyageurs montants dans les trains Transilien** est publié par SNCF sur [ressources.data.sncf.com](https://ressources.data.sncf.com/explore/dataset/comptage-voyageurs-trains-transilien/).

Il recense la fréquentation du réseau Transilien en Île-de-France.

Chaque enregistrement représente un comptage de voyageurs montants dans un train Transilien, pour :
- une **gare** donnée
- une **ligne** donnée (B, C, H, J, L, N, P, R, U...)
- un **type de jour** : jour ouvrable (JOB), samedi (SAM), dimanche (DIM)
- une **tranche horaire** donnée (ex : De 8h à 10h)

Les comptages sont réalisés soit automatiquement (capteurs embarqués), soit manuellement (une fois tous les 4 ans environ). Il n'y a pas d'historisation : chaque mise à jour remplace les données précédentes.

**Source :** SNCF Open Data — pas de clé API requise.

## Objectif

Ce notebook explore la mécanique d'appel à une API REST pour **récupérer dynamiquement les données Transilien**, en remplacement du téléchargement manuel d'un fichier CSV.

C'est la base de tout pipeline de données réel : au lieu d'une source statique et figée, on interroge directement la source, avec des filtres précis, à la demande.

Le code exploré ici sera ensuite encapsulé dans un script réutilisable : `fetch_transilien.py`.

In [1]:
import requests

url = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/comptage-voyageurs-trains-transilien/records"

params = {
    "limit": 5
}

response = requests.get(url, params=params)
print(response.status_code)
print(response.json())

200
{'total_count': 7328, 'results': [{'nom_gare': 'ABLON', 'code_gare': '87545269', 'type_jour': 'SAM', 'date': '2025-08-02', 'annee': '2025', 'ligne': 'C', 'axe': 'C', 'tranche_horaire': 'Après 20h', 'somme_de_montants': 118}, {'nom_gare': 'AEROPORT-CHARLES-DE-GAULLE-2-(TERMINAL-2)', 'code_gare': '87001479', 'type_jour': 'JOB', 'date': '2023-10-10', 'annee': '2023', 'ligne': 'B', 'axe': 'B', 'tranche_horaire': 'De 16h à 20h', 'somme_de_montants': 3868}, {'nom_gare': 'AEROPORT-CHARLES-DE-GAULLE-1-(TERMINAL-3)', 'code_gare': '87271460', 'type_jour': 'SAM', 'date': '2019-03-16', 'annee': '2019', 'ligne': 'B', 'axe': 'B', 'tranche_horaire': 'Avant 6h', 'somme_de_montants': 82}, {'nom_gare': 'AEROPORT-CHARLES-DE-GAULLE-1-(TERMINAL-3)', 'code_gare': '87271460', 'type_jour': 'DIM', 'date': '2019-03-17', 'annee': '2019', 'ligne': 'B', 'axe': 'B', 'tranche_horaire': 'Avant 6h', 'somme_de_montants': 70}, {'nom_gare': 'AEROPORT-CHARLES-DE-GAULLE-2-(TERMINAL-2)', 'code_gare': '87001479', 'type_j

## Lecture du résultat

Le `status_code` **200** confirme que la requête a abouti.

La réponse JSON contient deux clés :

- **`total_count`** : nombre total d'enregistrements disponibles dans le dataset (7328). C'est ce chiffre qui pilote la pagination.
- **`results`** : la liste des enregistrements demandés (ici 5).

Chaque enregistrement contient les champs suivants :

| Champ | Description |
|---|---|
| `nom_gare` | Nom de la gare |
| `code_gare` | Code UIC unique de la gare |
| `type_jour` | `JOB` = ouvrable · `SAM` = samedi · `DIM` = dimanche |
| `date` | Date du comptage |
| `ligne` | Ligne Transilien (B, C, H, J...) |
| `tranche_horaire` | Créneau horaire du comptage |
| `somme_de_montants` | Nombre de voyageurs montants |

In [2]:
import json

print(json.dumps(response.json(), indent=2, ensure_ascii=False))

{
  "total_count": 7328,
  "results": [
    {
      "nom_gare": "ABLON",
      "code_gare": "87545269",
      "type_jour": "SAM",
      "date": "2025-08-02",
      "annee": "2025",
      "ligne": "C",
      "axe": "C",
      "tranche_horaire": "Après 20h",
      "somme_de_montants": 118
    },
    {
      "nom_gare": "AEROPORT-CHARLES-DE-GAULLE-2-(TERMINAL-2)",
      "code_gare": "87001479",
      "type_jour": "JOB",
      "date": "2023-10-10",
      "annee": "2023",
      "ligne": "B",
      "axe": "B",
      "tranche_horaire": "De 16h à 20h",
      "somme_de_montants": 3868
    },
    {
      "nom_gare": "AEROPORT-CHARLES-DE-GAULLE-1-(TERMINAL-3)",
      "code_gare": "87271460",
      "type_jour": "SAM",
      "date": "2019-03-16",
      "annee": "2019",
      "ligne": "B",
      "axe": "B",
      "tranche_horaire": "Avant 6h",
      "somme_de_montants": 82
    },
    {
      "nom_gare": "AEROPORT-CHARLES-DE-GAULLE-1-(TERMINAL-3)",
      "code_gare": "87271460",
      "type_jour": "D

In [3]:
import pandas as pd

df = pd.DataFrame(response.json()["results"])
df

,nom_gare,code_gare,type_jour,date,annee,ligne,axe,tranche_horaire,somme_de_montants
0,ABLON,87545269,SAM,2025-08-02,2025,C,C,Après 20h,118
1,AEROPORT-CHARLES-DE-GAULLE-2-(TERMINAL-2),87001479,JOB,2023-10-10,2023,B,B,De 16h à 20h,3868
2,AEROPORT-CHARLES-DE-GAULLE-1-(TERMINAL-3),87271460,SAM,2019-03-16,2019,B,B,Avant 6h,82
3,AEROPORT-CHARLES-DE-GAULLE-1-(TERMINAL-3),87271460,DIM,2019-03-17,2019,B,B,Avant 6h,70
4,AEROPORT-CHARLES-DE-GAULLE-2-(TERMINAL-2),87001479,SAM,2019-03-23,2019,B,B,De 16h à 20h,3376


In [7]:
import requests
import json

url = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/comptage-voyageurs-trains-transilien/records"

params = {
    "limit": 5,
    "where": 'ligne="H"'
}

response = requests.get(url, params=params)
print(response.status_code)
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

200
{
  "total_count": 731,
  "results": [
    {
      "nom_gare": "CHAMP-DE-COURSES-D'ENGHIEN",
      "code_gare": "87276030",
      "type_jour": "DIM",
      "date": "2024-01-21",
      "annee": "2024",
      "ligne": "H",
      "axe": "H",
      "tranche_horaire": "De 10h à 16h",
      "somme_de_montants": 727
    },
    {
      "nom_gare": "CHAMP-DE-COURSES-D'ENGHIEN",
      "code_gare": "87276030",
      "type_jour": "SAM",
      "date": "2024-01-20",
      "annee": "2024",
      "ligne": "H",
      "axe": "H",
      "tranche_horaire": "Avant 6h",
      "somme_de_montants": 47
    },
    {
      "nom_gare": "CHAPONVAL",
      "code_gare": "87276162",
      "type_jour": "JOB",
      "date": "2024-01-16",
      "annee": "2024",
      "ligne": "H",
      "axe": "H",
      "tranche_horaire": "De 16h à 20h",
      "somme_de_montants": 14
    },
    {
      "nom_gare": "EPLUCHES",
      "code_gare": "87276147",
      "type_jour": "DIM",
      "date": "2024-01-21",
      "annee": "2024",

In [8]:
import requests
import json

total_count = 200
offset = 0
results = []

while offset < total_count:
      params = {
        "limit": 100,
        "where": 'ligne="H"',
        "offset": offset
      }
      url = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/comptage-voyageurs-trains-transilien/records"
      response = requests.get(url, params=params)
      results.append(response.json()["results"])
      offset += 100



print(response.status_code)
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

200
{
  "total_count": 731,
  "results": [
    {
      "nom_gare": "MERIEL",
      "code_gare": "87276675",
      "type_jour": "SAM",
      "date": "2024-01-20",
      "annee": "2024",
      "ligne": "H",
      "axe": "H",
      "tranche_horaire": "Après 20h",
      "somme_de_montants": 38
    },
    {
      "nom_gare": "MERY-SUR-OISE",
      "code_gare": "87276667",
      "type_jour": "SAM",
      "date": "2024-01-20",
      "annee": "2024",
      "ligne": "H",
      "axe": "H",
      "tranche_horaire": "Avant 6h",
      "somme_de_montants": 22
    },
    {
      "nom_gare": "MERY-SUR-OISE",
      "code_gare": "87276667",
      "type_jour": "SAM",
      "date": "2024-01-20",
      "annee": "2024",
      "ligne": "H",
      "axe": "H",
      "tranche_horaire": "De 6h à 10h",
      "somme_de_montants": 156
    },
    {
      "nom_gare": "MONTIGNY-BEAUCHAMP",
      "code_gare": "87276089",
      "type_jour": "DIM",
      "date": "2024-01-21",
      "annee": "2024",
      "ligne": "H",
  

In [10]:
import pandas as pd

all_results = [record for page in results for record in page]
df = pd.DataFrame(all_results)
df
df.shape

(200, 9)

## Bilan de l'exploration

Cette section d'exploration a couvert la mécanique fondamentale d'un appel API :
requête filtrée → pagination → JSON → DataFrame pandas.

La partie la plus abstraite est la pagination : on ne parcourt pas une liste visible,
on construit une boucle qui avance dans un dataset distant, page par page, sans jamais
le voir en entier. C'est difficile à visualiser, mais c'est précisément ce qui rend
la démarche puissante.

Passer par cette abstraction est nécessaire : c'est ce qui permet de remplacer
un téléchargement manuel figé par une récupération dynamique, paramétrable,
et intégrable dans un pipeline automatisé.

Le notebook a servi de terrain d'exploration. Le code produit ici sera maintenant
encapsulé dans un script réutilisable : `fetch_transilien.py`.